In [0]:
# ── Configuration ──────────────────────────────────────────────────────────────
dbutils.widgets.text("CATALOG", "dev")
CATALOG = dbutils.widgets.get("CATALOG")

In [0]:
# ── Step 1: Read Bronze, Cast Types, Drop Metadata Columns ──
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def typed(name, dtype="string"):
    """Safe cast: blank -> NULL, unparseable -> NULL (never throws). Returns a column named `name`."""
    if dtype == "string":
        return F.col(name).cast("string").alias(name)
    return F.expr(f"try_cast(nullif(trim(cast(`{name}` as string)), '') as {dtype})").alias(name)

bronze_df = spark.table(f"{CATALOG}.bronze.region")

# Metadata columns are NOT selected, so they are dropped.
# _row_seq remembers the source order so "keep the first occurrence" is deterministic on every rerun.
typed_df = (bronze_df.select(
    typed("r_regionkey", "int"),
    typed("r_name", "string"),
    typed("r_comment", "string")
    )
    .withColumn("_row_seq", F.monotonically_increasing_id()))

bronze_count = bronze_df.count()
print(f"Bronze region rows read: {bronze_count}")
display(typed_df.drop("_row_seq"))

In [0]:
# ── Step 2: Primary Key Validation ──
# Rule: r_regionkey must be NOT NULL and UNIQUE.
# Failure codes: NULL_PK, DUPLICATE_PK
# First occurrence (by source order) stays in Silver; later repeats are quarantined.

pk_null = F.col("r_regionkey").isNull()
pk_window = Window.partitionBy("r_regionkey").orderBy("_row_seq")      # NO "desc" and NO running count: plain row_number

flagged = (typed_df
    .withColumn("is_null_pk", pk_null)
    .withColumn("_pk_rn", F.when(pk_null, F.lit(0)).otherwise(F.row_number().over(pk_window)))
    .withColumn("is_duplicate_pk", F.col("_pk_rn") > 1))

print(f"NULL_PK failures:      {flagged.filter('is_null_pk').count()}")
print(f"DUPLICATE_PK failures: {flagged.filter('is_duplicate_pk').count()}")

In [0]:
# ── Step 3: Build Rejection Reasons, Split Clean vs Quarantine, Write Tables ──
flagged = flagged.withColumn(
    "rejection_reason",
    F.concat_ws(", ",
        F.when(F.col("is_null_pk"), F.lit("NULL_PK")),
        F.when(F.col("is_duplicate_pk"), F.lit("DUPLICATE_PK"))
    ),
)

out_cols = ["r_regionkey", "r_name", "r_comment"]

silver_df     = flagged.filter(F.col("rejection_reason") == "").select(*out_cols)
quarantine_df = flagged.filter(F.col("rejection_reason") != "").select(*out_cols, "rejection_reason")

silver_count = silver_df.count()
quarantine_count = quarantine_df.count()
print(f"Silver rows:     {silver_count}")
print(f"Quarantine rows: {quarantine_count}")

# overwrite => rerunning gives the same result
(silver_df.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.region"))
(quarantine_df.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.region_quarantine"))

print("Tables written: silver.region, silver.region_quarantine")

In [0]:
# ── Step 4: Bronze-Silver-Quarantine Reconciliation ──
bronze_total     = spark.table(f"{CATALOG}.bronze.region").count()
silver_total     = spark.table(f"{CATALOG}.silver.region").count()
quarantine_total = spark.table(f"{CATALOG}.silver.region_quarantine").count()

print(f"Bronze row count:      {bronze_total}")
print(f"Silver row count:      {silver_total}")
print(f"Quarantine row count:  {quarantine_total}")
print(f"Silver + Quarantine:   {silver_total + quarantine_total}")

assert bronze_total == silver_total + quarantine_total, (
    f"RECONCILIATION FAILED: Bronze ({bronze_total}) != Silver ({silver_total}) + Quarantine ({quarantine_total})"
)
print("Reconciliation PASSED: Bronze = Silver + Quarantine")

In [0]:
# ── Step 5: Data Quality Report ──
# Every check is listed, even when zero rows fail. Region has no FK rules.
rules = [
    ("NULL_PK", "r_regionkey must not be null", "is_null_pk"),
    ("DUPLICATE_PK", "r_regionkey must be unique (first occurrence kept)", "is_duplicate_pk")
]
dq_rows = [("region", code, desc, flagged.filter(F.col(flag)).count(), "Quarantine") for code, desc, flag in rules]

dq_report = spark.createDataFrame(
    dq_rows, ["table", "rule_code", "check_description", "failing_rows", "action_taken"]
)
display(dq_report)

# Persist so the whole run is auditable (overwrite per table is not needed: one table per notebook)
(dq_report.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.dq_report_region"))

if quarantine_count > 0:
    print(f"\n── Quarantined Records ({quarantine_count} rows) ──")
    display(spark.table(f"{CATALOG}.silver.region_quarantine"))
else:
    print("\nNo quarantined records — all rows passed every check.")

print(f"\n── Silver region ({silver_count} rows) ──")
display(spark.table(f"{CATALOG}.silver.region").limit(20))